# Retrieval Evaluation — Baseline Analysis

This notebook analyzes the baseline performance of the semantic retriever.

Baseline configuration:

- Embedding model: `paraphrase-multilingual-MiniLM-L12-v2`
- Knowledge base: `context.txt`
- Retrieval representation: manually defined search keys
- Retrieval threshold: 0.30
- Evaluation dataset: 100 manually curated queries

The analysis separates:
1. ranking performance,
2. retrieval/rejection decisions,
3. performance across query categories and difficulty levels,
4. out-of-domain behavior.

## Imports and loading data

We start importing the libraries:

In [2]:
import json
from pathlib import Path

import pandas as pd

We continue loading the results:

In [3]:
# Load results
RESULTS_PATH = Path("retrieval_results.json")

with RESULTS_PATH.open("r", encoding="utf-8") as file:
    results = json.load(file)

Check:

In [10]:
print("Type: ", type(results))
print("len: ", len(results))
# Example
results[0]

Type:  <class 'list'>
len:  100


{'case_id': 'fever_001',
 'query': 'Tengo fiebre',
 'expected_id': 'fever',
 'should_retrieve': True,
 'query_type': 'direct',
 'difficulty': 'easy',
 'predicted_id': 'fever',
 'best_score': 0.6667430400848389,
 'retrieved_id': 'fever',
 'expected_rank': 1,
 'retrieval_correct': True,
 'decision_correct': True,
 'ranking': [{'rank': 1, 'id': 'fever', 'score': 0.6667430400848389},
  {'rank': 2, 'id': 'respiratory', 'score': 0.6096863746643066},
  {'rank': 3, 'id': 'headache', 'score': 0.5492120385169983},
  {'rank': 4, 'id': 'abdominal', 'score': 0.4260847568511963},
  {'rank': 5, 'id': 'greeting', 'score': 0.24636825919151306},
  {'rank': 6, 'id': 'hours_location', 'score': 0.18111678957939148},
  {'rank': 7, 'id': 'trauma', 'score': 0.1654168963432312},
  {'rank': 8, 'id': 'appointments', 'score': 0.0965028926730156},
  {'rank': 9, 'id': 'out_of_domain', 'score': 0.011328473687171936},
  {'rank': 10, 'id': 'insurance_payments', 'score': -0.027254842221736908}]}

## Building the dataframe

In [11]:
df = pd.DataFrame(results)
df.head()

,case_id,query,expected_id,should_retrieve,query_type,difficulty,predicted_id,best_score,retrieved_id,expected_rank,retrieval_correct,decision_correct,ranking
0,fever_001,Tengo fiebre,fever,True,direct,easy,fever,0.666743,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.6667430..."
1,fever_002,Tengo 38.5 de temperatura,fever,True,direct,easy,fever,0.609955,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.6099554..."
2,fever_003,El termómetro me marca 38.7 desde anoche,fever,True,paraphrase,medium,fever,0.520986,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.5209861..."
3,fever_004,Estoy con escalofríos y siento el cuerpo muy c...,fever,True,symptom_description,medium,fever,0.711315,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.7113145..."
4,fever_005,Creo que levanté temperatura y estoy bastante ...,fever,True,paraphrase,medium,fever,0.584605,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.5846052..."


In [12]:
df.columns

Index(['case_id', 'query', 'expected_id', 'should_retrieve', 'query_type',
       'difficulty', 'predicted_id', 'best_score', 'retrieved_id',
       'expected_rank', 'retrieval_correct', 'decision_correct', 'ranking'],
      dtype='str')

## Sanity checks

We start with some sanity checks, in order to explore the dataframe.

In [13]:
df.shape

(100, 13)

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   case_id            100 non-null    str    
 1   query              100 non-null    str    
 2   expected_id        80 non-null     str    
 3   should_retrieve    100 non-null    bool   
 4   query_type         100 non-null    str    
 5   difficulty         100 non-null    str    
 6   predicted_id       100 non-null    str    
 7   best_score         100 non-null    float64
 8   retrieved_id       79 non-null     str    
 9   expected_rank      80 non-null     float64
 10  retrieval_correct  100 non-null    bool   
 11  decision_correct   100 non-null    bool   
 12  ranking            100 non-null    object 
dtypes: bool(3), float64(2), object(1), str(7)
memory usage: 8.2+ KB


In [15]:
df["difficulty"].value_counts()

difficulty
hard      39
medium    36
easy      25
Name: count, dtype: int64

In [16]:
df["query_type"].value_counts()

query_type
paraphrase                20
direct                    18
unknown_out_of_domain     10
unsupported_medical       10
symptom_description        9
colloquial                 9
explicit_out_of_domain     8
noisy                      7
intent_description         4
indirect                   3
greeting_with_noise        2
Name: count, dtype: int64

In [17]:
df["expected_id"].value_counts(dropna=False)

expected_id
NaN                   20
fever                  8
headache               8
respiratory            8
abdominal              8
trauma                 8
appointments           8
hours_location         8
insurance_payments     8
greeting               8
out_of_domain          8
Name: count, dtype: int64

In [18]:
df.groupby(["should_retrieve", "difficulty"]).size()

should_retrieve  difficulty
False            easy           5
                 hard           9
                 medium         6
True             easy          20
                 hard          30
                 medium        30
dtype: int64

We can see some easy metrics.

In [19]:
df["retrieval_correct"].mean()

np.float64(0.63)

In [20]:
df["decision_correct"].mean()

np.float64(0.79)

Remember that `retrieval_correct` tells us whether the system returned the correct ID
(or correctly returned no ID), while `decision_correct` only tells us whether the
system made the correct retrieve-vs-reject decision, regardless of which ID was retrieved.